# Keyword context / KWIC export (v4)

Rebuilt 2026-09-25 in response to actual HTRC review feedback on the v3 release (Ryan,
HTRC): v3's window-merging let densely-clustered keywords chain into arbitrarily large
passages -- "full pages" in Ryan's words -- which broke the "defined number of sentences
adjacent to a keyword" standard HTRC expects for KWIC exports. v4 fixes this at the root
rather than patching v3:

1. **No merging.** Every keyword occurrence gets its own independent, fixed-size snippet
   -- 1 sentence before + the keyword sentence + 1 sentence after, always exactly 3
   sentences (fewer only at a volume's first/last sentence). Two different keywords in
   the same sentence produce two separate rows -- duplicate text across rows is fine,
   since each keyword is modeled independently, not as a merged, overlapping mass.
2. **Fully randomized row order** (global shuffle, fixed seed for reproducibility) --
   same rationale as v3: exported rows shouldn't be assemblable into a sequential
   reading of any one book.
3. **A per-novel, per-lexicon 40% check, done iteratively, on ACCURATE coverage.**
   Each book/lexicon is extracted uncapped first. Percentage is computed from the
   *union* of covered sentences, not the sum of each snippet's word count -- two
   snippets that share a sentence (e.g. two keywords in one sentence, or two nearby
   keyword hits whose windows overlap) would otherwise have that sentence's words
   counted twice, overstating real exposure. If the accurate percentage is >= 40%, the
   number of snippets kept per keyword is halved (evenly sampled) and rechecked --
   repeating until it's under 40%, or until the per-keyword cap hits a floor of 1 (can't
   reduce further without dropping a keyword's presence in that book entirely, in which
   case it's flagged rather than silently forced under the line -- Dez has no capacity
   for manual review of every capsule run, so this is informational, not blocking).
4. **A second check on the env+tech UNION**, after each lexicon's own reduction. Two
   lexicons could each individually sit under 40%, but if the same passages trigger
   both (e.g. "nuclear reactor" hits both nuclear_atomic and reactor), their combined
   unique exported text could still exceed 40%. If so, the same halving reduction is
   applied across the combined pool, split back by lexicon afterward.
5. **Every excerpt notes which novel it came from** (title/author/year/era/htid) --
   deliberately not *where* in the novel (no sentence-position field, tracked only
   internally for the coverage calculations above, never exported), since noting the
   source book doesn't require also encoding reconstructable position, and that's the
   same category of concern that came up separately on v3's design.

No cap on file size beyond the existing rolling-CSV split (before ~900MB, under the
capsule's 1GB export limit) -- Dez is not concerned about total export volume, just
per-novel exposure and passage size.

Must run **inside the capsule, in secure mode**.

## Imports

In [ ]:
import csv
import json
import os
import random
import re
from collections import Counter
from multiprocessing import get_context
from pathlib import Path

import pandas as pd
from nltk.tokenize import TreebankWordTokenizer, sent_tokenize

## Check the sentence tokenizer is available

Secure mode has no network -- if this fails, run
`python3 -c "import nltk; nltk.download('punkt'); nltk.download('punkt_tab')"` from
maintenance mode first, then switch to secure mode and retry.

In [ ]:
try:
    sent_tokenize("Checking that the sentence tokenizer's data is available. This is only a test.")
    print("tokenizer OK")
except LookupError as e:
    raise SystemExit(
        "NLTK sentence-tokenizer data not found, and secure mode has no network to fetch it. "
        "Run this from maintenance mode BEFORE switching to secure mode:\n"
        "  python3 -c \"import nltk; nltk.download('punkt'); nltk.download('punkt_tab')\"\n"
        f"Original error: {e}"
    )

## OCR cleaning

Copied verbatim from `SF_word2vec_eras_v2.ipynb` (cell `46a9ca82`).

In [ ]:
_norm_ws = re.compile(r"\s+")
_only_number = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
_hyphen_break = re.compile(r"(\w)-\s*\n\s*(\w)")
_word = re.compile(r"[A-Za-z']+")


def discover_volumes(base_dir, ids=None):
    vols = {}
    for d in sorted(p for p in Path(base_dir).iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols


def _norm_line(line):
    return _norm_ws.sub(" ", re.sub(r"\d+", "", line)).strip().lower()


def _volume_vocab(page_paths):
    words = set()
    for p in page_paths:
        text, _ = _hyphen_break.subn(r"\1\2", p.read_text(encoding="utf-8", errors="replace"))
        words.update(w.lower() for w in _word.findall(text) if len(w) > 1)
    return words


def build_dictionary(volumes, min_vols, procs=None):
    df = Counter()
    with get_context("fork").Pool(procs or min(16, os.cpu_count() or 1)) as pool:
        for vocab in pool.imap_unordered(_volume_vocab, list(volumes.values()), chunksize=8):
            df.update(vocab)
    return {w for w, c in df.items() if c >= min_vols}


def page_dict_rate(text, dictionary):
    words = [w.lower() for w in _word.findall(text) if len(w) > 1]
    return sum(1 for w in words if w in dictionary) / len(words) if words else 0.0


def clean_volume(page_paths, dictionary, running_head_min_pages, running_head_frac,
                  running_head_max_chars, page_min_dict_rate):
    pages = [p.read_text(encoding="utf-8", errors="replace") for p in page_paths]
    line_pages = Counter()
    per_page_lines = []
    for text in pages:
        lines = text.split("\n")
        per_page_lines.append(lines)
        line_pages.update({_norm_line(l) for l in lines if 0 < len(l.strip()) <= running_head_max_chars})
    thresh = max(running_head_min_pages, int(running_head_frac * len(pages)))
    heads = {l for l, c in line_pages.items() if c >= thresh and l}
    kept = []
    for lines in per_page_lines:
        out = []
        for l in lines:
            if (len(l.strip()) <= running_head_max_chars and _norm_line(l) in heads) or _only_number.match(l):
                continue
            out.append(l)
        page_text, _ = _hyphen_break.subn(r"\1\2", "\n".join(out))
        if dictionary is not None and page_min_dict_rate and page_dict_rate(page_text, dictionary) < page_min_dict_rate:
            continue
        kept.append(page_text)
    text, _ = _hyphen_break.subn(r"\1\2", "\n".join(kept))
    return text


def load_docs_keep_id(base_dir, dictionary, ids=None, **clean_kwargs):
    docs = {}
    for htid, pages in discover_volumes(base_dir, ids).items():
        docs[htid] = clean_volume(pages, dictionary, **clean_kwargs)
    return docs

## Tokenization for keyword matching

Same `TOKEN_RE` as `SF_word2vec_eras_v2.ipynb` (cell `be5f5689`).

In [ ]:
_tokenizer = TreebankWordTokenizer()
TOKEN_RE = re.compile(r"^[a-z]+(?:'[a-z]+)?$")


def clean_tokens(tokens):
    return [t for t in tokens if TOKEN_RE.match(t) and len(t) > 1]


def sentences_with_tokens(text):
    """Raw (readable, original-case) sentences paired with their lowercased match-tokens."""
    raw_sents = sent_tokenize(text)
    out = []
    for s in raw_sents:
        toks = set(clean_tokens(_tokenizer.tokenize(s.lower())))
        out.append((s, toks))
    return out

## Era assignment

In [ ]:
def assign_era(year, cutoffs=(1962, 1972), labels=("era_a", "era_b", "era_c")):
    for cutoff, label in zip(cutoffs, labels):
        if year < cutoff:
            return label
    return labels[-1]

## Metadata loading

Same Ace-Doubles handling as the other notebooks -- `year` identical within every
duplicate-`htid` group (checked), `title`/`author` joined with `" / "`.

In [ ]:
def load_metadata(path):
    df = pd.read_csv(path)
    df["htid"] = df["htid"].astype(str)

    def join_unique(values):
        return " / ".join(dict.fromkeys(str(v) for v in values))

    grouped = df.groupby("htid").agg(
        title=("title", join_unique),
        author=("author", join_unique),
        year=("year", "first"),
    )
    return grouped.to_dict("index")

## Lexicons (reviewed 2026-09-23)

Environmental: 101 words. Technology: 24 words, zero overlap. Unchanged from v2/v3.

In [ ]:
ENV_WORD_GROUPS = {
    "landscape_baseline": ["river", "creek", "stream", "water", "forest", "nature", "wilderness", "jungle",
                            "ocean", "landscape", "levee", "dam", "reservoir", "estuary", "wetland",
                            "marsh", "watershed"],
    "ecology_concept": ["ecology", "ecosystem", "environment", "biosphere", "habitat", "balance"],
    "contamination": ["contamination", "waste", "smog", "fumes", "chemical", "pesticide", "insecticide",
                       "pollutant", "exhaust", "toxic", "polluted", "pollution"],
    "waste_infrastructure": ["sewer", "sewage", "drainage", "effluent", "runoff", "wastewater",
                              "cesspool", "sludge", "septic", "plumbing"],
    "population_scarcity": ["overpopulation", "population", "famine", "scarcity", "starvation", "resource", "drought"],
    "energy": ["oil", "fuel", "energy", "coal"],
    "nuclear_atomic": ["radiation", "radioactive", "fallout", "nuclear", "atomic", "bomb", "meltdown"],
    "cosmic_natural_causation": ["solar", "cosmic", "celestial", "geological", "planetary"],
    "human_agency": ["mankind", "humanity", "civilization", "industrial", "war"],
    "disaster_collapse": ["wasteland", "extinction", "collapse", "barren", "dying", "decay", "catastrophe",
                           "apocalypse", "plague"],
    "climate_weather": ["climate", "weather", "warming", "greenhouse", "atmosphere", "temperature",
                         "flood", "flooding", "storm", "hurricane", "glacier", "carbon", "ozone"],
    "space_earth_framing": ["earth", "homeworld", "colony", "frontier", "terraform", "alien"],
}

TECH_WORD_GROUPS = {
    "automation_machinery": ["machinery", "mechanical", "automaton", "automation", "automated"],
    "artificial_beings": ["robot", "android", "cyborg"],
    "computing_electronics": ["computer", "cybernetic", "electronic", "circuitry"],
    "engineering_industry": ["engineering", "engineer", "technology", "technological", "factory"],
    "synthetic_material": ["synthetic", "artificial"],
    "space_energy_tech": ["rocket", "spacecraft", "satellite", "laser", "reactor"],
}


def word_to_group_map(word_groups):
    return {w: g for g, ws in word_groups.items() for w in ws}


env_words = set(word_to_group_map(ENV_WORD_GROUPS))
tech_words = set(word_to_group_map(TECH_WORD_GROUPS))
env_word_to_group = word_to_group_map(ENV_WORD_GROUPS)
tech_word_to_group = word_to_group_map(TECH_WORD_GROUPS)
print(f"env: {len(env_words)} words, tech: {len(tech_words)} words, overlap: {env_words & tech_words}")

## Snippet extraction -- no merging

One row per keyword *occurrence*, always a fixed 3-sentence window (1 before + the
keyword sentence + 1 after). Two different keywords in the same sentence produce two
separate rows. This is the direct fix for HTRC's "full pages" finding on v3 -- no window
can ever exceed this fixed size, regardless of how densely keywords cluster.

In [ ]:
def extract_snippets(sent_list, lexicon_words, word_to_group, sentences_before=1, sentences_after=1):
    """sent_list: [(raw_sentence, token_set), ...] for one novel.
    Returns list of dicts: one per keyword occurrence (not merged). Each snippet also
    carries its underlying sentence range (_start/_end, leading underscore) -- used only
    internally to compute accurate, non-double-counted coverage percentages below;
    stripped before anything is written to the exported CSV."""
    n = len(sent_list)
    snippets = []
    for i, (_, toks) in enumerate(sent_list):
        matched = toks & lexicon_words
        if not matched:
            continue
        start = max(0, i - sentences_before)
        end = min(n - 1, i + sentences_after)
        context = " ".join(sent_list[j][0] for j in range(start, end + 1))
        context_words = len(context.split())
        for w in matched:
            snippets.append({
                "keyword": w,
                "group": word_to_group[w],
                "context": context,
                "context_words": context_words,
                "_start": start,
                "_end": end,
            })
    return snippets

## Per-novel 40% check, with iterative per-keyword halving

`unique_word_coverage()` computes the real, non-double-counted percentage: the union of
every kept snippet's sentence range, each sentence counted once no matter how many
snippets touch it. Summing each snippet's own word count instead would double-count any
sentence shared by two snippets (two keywords in one sentence, or two nearby hits whose
windows overlap) -- always an overestimate, never an undercount, but not the true number.

Uncapped first. If the accurate percentage is >= `FLAG_THRESHOLD_PCT`, the per-keyword
snippet cap is set to half the largest keyword count present and applied (evenly
sampled, so a halved keyword's remaining snippets still spread across the book rather
than clustering at the start) -- then rechecked, halving again each time it's still
over, until either it's under threshold or the cap hits a floor of 1 (can't reduce a
keyword below one snippet without dropping it from that book entirely, at which point
it's left as-is and flagged).

In [ ]:
def _evenly_sample(items, k):
    """k evenly-spaced items from a list (first and last always included).
    Returns the list unchanged if it already has k or fewer items."""
    n = len(items)
    if n <= k:
        return items
    if k <= 1:
        return items[:1]
    picked = sorted({round(i * (n - 1) / (k - 1)) for i in range(k)})
    return [items[i] for i in picked]


def _cap_per_keyword(snippets, cap):
    by_word = {}
    for s in snippets:
        by_word.setdefault(s["keyword"], []).append(s)
    out = []
    for w, items in by_word.items():
        out.extend(_evenly_sample(items, cap) if len(items) > cap else items)
    return out


def unique_word_coverage(snippets, sent_list):
    """Accurate exported-word count: the union of every snippet's sentence range,
    each sentence counted once -- not the sum of (possibly overlapping) per-snippet
    context_words, which double-counts any sentence covered by more than one snippet."""
    covered = set()
    for s in snippets:
        covered.update(range(s["_start"], s["_end"] + 1))
    return sum(len(sent_list[i][0].split()) for i in covered)


def reduce_until_under_threshold(snippets, sent_list, book_total_words, threshold_pct, floor=1):
    """Returns (final_snippets, final_pct, rounds_halved, still_over). Works on any list
    of snippet dicts with _start/_end/keyword -- used both per-lexicon and, if needed,
    on the combined env+tech pool for the cross-lexicon check below."""
    def pct_of(rows):
        return 100 * unique_word_coverage(rows, sent_list) / book_total_words if book_total_words else 0.0

    current = snippets
    pct = pct_of(current)
    if pct < threshold_pct or not snippets:
        return current, pct, 0, False

    counts = Counter(s["keyword"] for s in snippets)
    cap = max(counts.values())
    rounds = 0
    while pct >= threshold_pct and cap > floor:
        cap = max(floor, cap // 2)
        current = _cap_per_keyword(snippets, cap)
        pct = pct_of(current)
        rounds += 1

    return current, pct, rounds, pct >= threshold_pct

## Config

In [ ]:
TEXT_DIR = "/media/secure_volume/fa50b375-3216-4edd-a685-98488562b723"
METADATA_CSV = "/home/dcuser/Desktop/Clifi-htrc/notebooks/metadata_august2026.csv"
OUT_DIR = Path("/media/secure_volume/out_keyword_context_v4")
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

SENTENCES_BEFORE = 1   # HTRC feedback on v3: keep this fixed and small, no merging
SENTENCES_AFTER = 1
FLAG_THRESHOLD_PCT = 40.0
REDUCTION_FLOOR = 1     # minimum snippets per keyword when reducing -- can't go below this
SHUFFLE_SEED = 20260925
MAX_CSV_BYTES = 900_000_000   # split into a new part file before hitting this, under the 1GB export limit
DICT_MIN_VOLS = 10
PAGE_MIN_DICT_RATE = 0.55
RUNNING_HEAD_MIN_PAGES = 3
RUNNING_HEAD_FRAC = 0.05
RUNNING_HEAD_MAX_CHARS = 60

## Load metadata

In [ ]:
meta_by_id = load_metadata(METADATA_CSV)
print(f"{len(meta_by_id):,} unique htids after grouping Ace Doubles / omnibus scans")

## Discover volumes + build corpus dictionary

In [ ]:
all_volumes = discover_volumes(TEXT_DIR)
print(f"found {len(all_volumes):,} volumes")

dictionary = build_dictionary(all_volumes, DICT_MIN_VOLS)
print(f"dictionary: {len(dictionary):,} words appear in >= {DICT_MIN_VOLS} volumes")

## Clean every volume -- THE SLOW STEP

Once this finishes, `docs` stays in kernel memory. Everything below is much cheaper and
safe to re-run without paying this cost again.

In [ ]:
docs = load_docs_keep_id(
    TEXT_DIR, dictionary,
    running_head_min_pages=RUNNING_HEAD_MIN_PAGES,
    running_head_frac=RUNNING_HEAD_FRAC,
    running_head_max_chars=RUNNING_HEAD_MAX_CHARS,
    page_min_dict_rate=PAGE_MIN_DICT_RATE,
)
print(f"loaded {len(docs):,} cleaned documents")

## Extract + reduce snippets from the already-cleaned text

Sentence-splits + extracts fixed-window snippets (no merging) for every novel, applies
the 40%-threshold reduction loop per book per lexicon, then checks the env+tech
*combined* union against the same threshold and reduces further if needed. Much cheaper
than the cleaning step above, so re-running from here after a failure is a small cost.

In [ ]:
env_rows = []
tech_rows = []
book_summary_rows = []
flagged_books = []

for count, htid in enumerate(docs, 1):
    text = docs[htid]
    book_total_words = len(text.split())
    if book_total_words == 0:
        continue

    sent_list = sentences_with_tokens(text)
    info = meta_by_id.get(htid, {})
    year = info.get("year")
    row_meta = {
        "htid": htid,
        "title": info.get("title"),
        "author": info.get("author"),
        "year": year,
        "era": assign_era(year) if pd.notna(year) else None,
    }
    book_row = {**row_meta, "book_total_words": book_total_words}

    # Phase 1: each lexicon reduced independently, on its own accurate coverage.
    per_lexicon_stats = {}
    final_by_label = {}
    for label, words, word_to_group in [
        ("env", env_words, env_word_to_group),
        ("tech", tech_words, tech_word_to_group),
    ]:
        raw_snippets = extract_snippets(sent_list, words, word_to_group, SENTENCES_BEFORE, SENTENCES_AFTER)
        final_snippets, pct, rounds_halved, still_flagged = reduce_until_under_threshold(
            raw_snippets, sent_list, book_total_words, FLAG_THRESHOLD_PCT, REDUCTION_FLOOR
        )
        final_by_label[label] = final_snippets
        per_lexicon_stats[label] = {"raw": len(raw_snippets), "rounds": rounds_halved,
                                     "pct": pct, "flagged": still_flagged}
        if still_flagged:
            flagged_books.append((label, row_meta["title"], row_meta["author"], row_meta["year"], pct))

    # Phase 2: the env+tech UNION could still exceed the threshold even if each lexicon
    # looks fine alone (e.g. "nuclear reactor" hitting both nuclear_atomic and reactor).
    combined = final_by_label["env"] + final_by_label["tech"]
    combined_pct = 100 * unique_word_coverage(combined, sent_list) / book_total_words if book_total_words else 0.0
    combined_rounds = 0
    combined_flagged = False
    if combined_pct >= FLAG_THRESHOLD_PCT:
        combined, combined_pct, combined_rounds, combined_flagged = reduce_until_under_threshold(
            combined, sent_list, book_total_words, FLAG_THRESHOLD_PCT, REDUCTION_FLOOR
        )
        final_by_label["env"] = [s for s in combined if s["keyword"] in env_words]
        final_by_label["tech"] = [s for s in combined if s["keyword"] in tech_words]
        if combined_flagged:
            flagged_books.append(("combined", row_meta["title"], row_meta["author"], row_meta["year"], combined_pct))

    for label in ("env", "tech"):
        stats = per_lexicon_stats[label]
        book_row[f"{label}_snippets_raw"] = stats["raw"]
        book_row[f"{label}_snippets_final"] = len(final_by_label[label])   # reflects phase 2 too, if it ran
        book_row[f"{label}_pct_exported"] = round(stats["pct"], 2)         # this lexicon's own pct (phase 1)
        book_row[f"{label}_rounds_halved"] = stats["rounds"]
        book_row[f"{label}_flagged"] = stats["flagged"]
    book_row["combined_pct_exported"] = round(combined_pct, 2)
    book_row["combined_rounds_halved"] = combined_rounds
    book_row["combined_flagged"] = combined_flagged

    for label, out_rows in [("env", env_rows), ("tech", tech_rows)]:
        for s in final_by_label[label]:
            clean_s = {k: v for k, v in s.items() if not k.startswith("_")}
            out_rows.append({**clean_s, **row_meta})

    book_summary_rows.append(book_row)
    if count % 500 == 0:
        print(f"  ... {count:,}/{len(docs):,} novels processed")

print(f"done: {len(env_rows):,} env snippets, {len(tech_rows):,} tech snippets, "
      f"{len(book_summary_rows):,} books summarized")

## Print flagged books

Still over 40% even after halving down to the floor -- review before releasing.

In [ ]:
n_reduced = sum(1 for r in book_summary_rows
                if r["env_rounds_halved"] > 0 or r["tech_rounds_halved"] > 0 or r["combined_rounds_halved"] > 0)
n_combined_reduced = sum(1 for r in book_summary_rows if r["combined_rounds_halved"] > 0)
print(f"{n_reduced:,} books had at least one lexicon (or the combined pool) reduced by the halving loop "
      f"({n_combined_reduced:,} needed the combined-pool phase specifically).")

if flagged_books:
    print(f"*** {len(flagged_books)} lexicon/book combinations STILL >= {FLAG_THRESHOLD_PCT}% "
          f"even at the {REDUCTION_FLOOR}-per-keyword floor -- REVIEW BEFORE RELEASING: ***")
    for label, title, author, year, pct in sorted(flagged_books, key=lambda x: -x[4]):
        print(f"  [{label}] {pct:.1f}%  {title} ({author}, {year})")
else:
    print(f"no book exceeded {FLAG_THRESHOLD_PCT}% for either lexicon, or their combined union, after reduction.")

## Which keywords are driving the flagged books?

Small aggregate table (keyword + a count, no text) -- safe to export on its own, same as
the equivalent table in v3.

In [ ]:
combined_flagged_htids = {r["htid"] for r in book_summary_rows if r["combined_flagged"]}
flagged_htids = {
    "env": {r["htid"] for r in book_summary_rows if r["env_flagged"]} | combined_flagged_htids,
    "tech": {r["htid"] for r in book_summary_rows if r["tech_flagged"]} | combined_flagged_htids,
}

flagged_word_rows = []
for lexicon, rows, flagged_ids in [("env", env_rows, flagged_htids["env"]), ("tech", tech_rows, flagged_htids["tech"])]:
    counts = Counter(r["keyword"] for r in rows if r["htid"] in flagged_ids)
    for word, n in counts.items():
        flagged_word_rows.append({"lexicon": lexicon, "word": word, "snippets_in_flagged_books": n})

flagged_word_counts_df = pd.DataFrame(flagged_word_rows, columns=["lexicon", "word", "snippets_in_flagged_books"])
flagged_word_counts_df = flagged_word_counts_df.sort_values(
    ["lexicon", "snippets_in_flagged_books"], ascending=[True, False]
)
print(f"{len(flagged_htids['env'])} flagged env books, {len(flagged_htids['tech'])} flagged tech books")
print(flagged_word_counts_df.to_string(index=False))

In [ ]:
flagged_word_counts_path = OUT_DIR / "tables" / "flagged_word_counts.csv"
flagged_word_counts_df.to_csv(flagged_word_counts_path, index=False)
print(f"wrote {flagged_word_counts_path}")

reread_fwc = pd.read_csv(flagged_word_counts_path)
assert len(reread_fwc) == len(flagged_word_counts_df), (
    f"MISMATCH: wrote {len(flagged_word_counts_df)} rows but disk shows {len(reread_fwc)}"
)
print(f"VERIFIED: {flagged_word_counts_path} has {len(reread_fwc):,} rows on disk, matches expected {len(flagged_word_counts_df):,}.")

## Write the CSVs (auto-splitting) + the book summary

Rows are shuffled globally (fixed seed) before writing -- same rationale as v3: exported
rows shouldn't be assemblable into a sequential reading of any one book. No sentence-
position field is included (deliberately -- see the notebook overview).

Safe to re-run this cell (and the verify cell after it) as many times as needed --
`env_rows`, `tech_rows`, and `book_summary_rows` are already in memory.

In [ ]:
random.Random(SHUFFLE_SEED).shuffle(env_rows)
random.Random(SHUFFLE_SEED).shuffle(tech_rows)

CONTEXT_FIELDNAMES = ["keyword", "group", "context", "context_words", "htid", "title", "author", "year", "era"]


def write_rolling_csv(rows, out_dir, base_name, fieldnames, max_bytes):
    """Writes rows across part files (part1, part2, ...), splitting before max_bytes."""
    part = 1
    parts_written = []

    def open_part(n):
        path = out_dir / f"{base_name}_part{n}.csv"
        f = open(path, "w", newline="", encoding="utf-8")
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        parts_written.append(path)
        return f, writer

    f, writer = open_part(part)
    for row in rows:
        writer.writerow(row)
        if f.tell() >= max_bytes:
            f.close()
            part += 1
            f, writer = open_part(part)
    f.close()
    return parts_written


env_parts = write_rolling_csv(env_rows, OUT_DIR, "environment_context", CONTEXT_FIELDNAMES, MAX_CSV_BYTES)
print(f"environment_context: {len(env_rows):,} snippets across {len(env_parts)} file(s)")
for p in env_parts:
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")

tech_parts = write_rolling_csv(tech_rows, OUT_DIR, "technology_context", CONTEXT_FIELDNAMES, MAX_CSV_BYTES)
print(f"technology_context: {len(tech_rows):,} snippets across {len(tech_parts)} file(s)")
for p in tech_parts:
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")

summary_df = pd.DataFrame(book_summary_rows)
summary_path = OUT_DIR / "tables" / "book_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"wrote {summary_path}: {len(summary_df):,} rows")

manifest = {
    "n_volumes": len(docs),
    "sentences_before": SENTENCES_BEFORE,
    "sentences_after": SENTENCES_AFTER,
    "flag_threshold_pct": FLAG_THRESHOLD_PCT,
    "reduction_floor": REDUCTION_FLOOR,
    "cross_lexicon_union_checked": True,
    "shuffle_seed": SHUFFLE_SEED,
    "env_lexicon_size": len(env_words),
    "tech_lexicon_size": len(tech_words),
    "env_snippets": len(env_rows),
    "tech_snippets": len(tech_rows),
    "n_flagged": len(flagged_books),
}
manifest_path = OUT_DIR / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"wrote {manifest_path}")

## Verify every file actually landed on disk

Re-reads each file fresh from disk and checks it against the in-memory data.

In [ ]:
def verify_csv_parts(parts, expected_total_rows, label):
    total = 0
    for p in parts:
        assert p.exists(), f"MISSING: {p} was supposed to be written but does not exist on disk"
        total += len(pd.read_csv(p))
    assert total == expected_total_rows, (
        f"MISMATCH for {label}: wrote {expected_total_rows} rows but disk shows {total}"
    )
    print(f"VERIFIED: {label} has {total:,} rows on disk across {len(parts)} file(s), matches expected {expected_total_rows:,}.")


verify_csv_parts(env_parts, len(env_rows), "environment_context")
verify_csv_parts(tech_parts, len(tech_rows), "technology_context")

reread_summary = pd.read_csv(summary_path)
assert len(reread_summary) == len(summary_df), (
    f"MISMATCH: book_summary.csv wrote {len(summary_df)} rows but disk shows {len(reread_summary)}"
)
print(f"VERIFIED: {summary_path} has {len(reread_summary):,} rows on disk, matches expected {len(summary_df):,}.")

reread_manifest = json.loads(manifest_path.read_text())
assert reread_manifest == manifest, "MISMATCH: manifest on disk does not match what was written"
print(f"VERIFIED: {manifest_path} matches on disk.")

## Release

Only run this once every cell above has printed `VERIFIED`, and after reviewing the
flagged-books list above. `add` and `done` are kept in separate cells so you can read
the `add` output before committing to `done`.

In [ ]:
import subprocess


def run_releaseresults(*args):
    cmd = ["releaseresults", *args]
    print("$ " + " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"releaseresults exited with code {result.returncode}")
    return result

In [ ]:
run_releaseresults("add", str(OUT_DIR / "tables"),
                    *[str(p) for p in env_parts], *[str(p) for p in tech_parts],
                    str(OUT_DIR / "MANIFEST.json"))

Check the output above looks right, then run this to finalize:

In [ ]:
run_releaseresults("done")